# 井上 ultrafine 残差直接贴片实验

本实验只回答一个问题：把旧 `enhance` 的 `well_patch` 动作直接作用于 `broad+detail`，视觉和正演结果会变成什么样。

不做检索、不训练网络、不优化贴片。供体残差随机选井和窗口，随后去均值、随机缩放、再次高通并做边缘 taper，最后直接加到目标 log-AI。

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy.ndimage import gaussian_filter1d

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').is_dir():
    repo_root = repo_root.parent
if not (repo_root / 'src').is_dir():
    raise RuntimeError('Could not locate repository root containing src/.')
src_root = repo_root / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.physics.numpy_backend import forward_depth, velocity_from_ai
from cup.synthetic.core.signal import finite_support_fir, valid_filter_decimate

SOURCE_DIR = (
    repo_root / 'note' / 'summary' / 'final_audit'
    / '20260810_well_prior_texture_and_decoder_failure'
    / 'results' / 'well_residual_decomposition'
    / '20260810_body_scale_decomposition'
)
OUTPUT_DIR = (
    repo_root / 'experiments' / 'well_residual_direct_patch'
    / 'results' / '20260811_direct_patch'
)
FIGURE_DIR = OUTPUT_DIR / 'figures'
ARTIFACT_DIR = OUTPUT_DIR / 'wells'
for directory in (OUTPUT_DIR, FIGURE_DIR, ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CANDIDATES = {
    'F10_B50': {'fine_fwhm_m': 10.0, 'key': 'f10_b50'},
    'F15_B50': {'fine_fwhm_m': 15.0, 'key': 'f15_b50'},
}
TARGET_WELLS = ('2-ANP-2A-RJS', 'L1-NW1', 'L5-NW5', 'L9-NW4A', 'NW11')
PATCH_SCALE_RANGE = (0.35, 0.80)
EDGE_TAPER_FRACTION = 0.10
RANDOM_SEED = 20260811
MODEL_GRID_INTERVAL_M = 5.0
FORWARD_OUTPUT_CHUNK_SIZE = 32
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 180, 'axes.grid': True, 'grid.alpha': 0.2})
print(f'Source: {SOURCE_DIR}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
required = [SOURCE_DIR / 'manifest.json', SOURCE_DIR / 'event_candidate_metrics.csv']
required.extend(SOURCE_DIR / 'wells' / f'{name}.npz' for name in TARGET_WELLS)
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)

source_manifest = json.loads((SOURCE_DIR / 'manifest.json').read_text(encoding='utf-8'))
event_metrics = pd.read_csv(SOURCE_DIR / 'event_candidate_metrics.csv')
wells = {}
for path in sorted((SOURCE_DIR / 'wells').glob('*.npz')):
    with np.load(path, allow_pickle=False) as artifact:
        wells[path.stem] = {key: artifact[key].copy() for key in artifact.files}

forward_input_path = repo_root / 'scripts' / 'output' / 'depth_forward_model_inputs_20260719_172553' / 'forward_model_inputs.json'
forward_inputs = json.loads(forward_input_path.read_text(encoding='utf-8'))
wavelet_path = repo_root / forward_inputs['wavelet']['path']
wavelet = pd.read_csv(wavelet_path)
wavelet_time_s = wavelet['time_s'].to_numpy(dtype=np.float64)
wavelet_amplitude = wavelet['amplitude'].to_numpy(dtype=np.float64)
AI_VP_A = float(forward_inputs['ai_velocity_relation']['a'])
AI_VP_B = float(forward_inputs['ai_velocity_relation']['b'])

all_log_ai = np.concatenate([
    item['well_log_ai'][item['well_valid'] & np.isfinite(item['well_log_ai'])]
    for item in wells.values()
])
LOG_AI_MIN = float(np.min(all_log_ai))
LOG_AI_MAX = float(np.max(all_log_ai))
print(f'Loaded wells: {sorted(wells)}')
print(f'Published log-AI range: [{LOG_AI_MIN:.3f}, {LOG_AI_MAX:.3f}]')

In [ ]:
def finite_runs(mask):
    values = np.asarray(mask, dtype=bool)
    padded = np.concatenate(([False], values, [False]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    return tuple(slice(int(start), int(stop)) for start, stop in changes.reshape(-1, 2))


def rms(values):
    finite = np.asarray(values, dtype=np.float64)
    finite = finite[np.isfinite(finite)]
    return float(np.sqrt(np.mean(finite**2))) if finite.size else np.nan


def safe_corr(left, right, support):
    valid = np.asarray(support, dtype=bool) & np.isfinite(left) & np.isfinite(right)
    if np.count_nonzero(valid) < 3:
        return np.nan
    a = np.asarray(left, dtype=np.float64)[valid]
    b = np.asarray(right, dtype=np.float64)[valid]
    if np.std(a) <= 1.0e-12 or np.std(b) <= 1.0e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def edge_taper(length, fraction=EDGE_TAPER_FRACTION):
    if length < 2:
        return np.ones(length, dtype=np.float64)
    edge = min(length // 2, max(2, int(round(length * fraction))))
    taper = np.ones(length, dtype=np.float64)
    ramp = np.sin(np.linspace(0.0, np.pi / 2.0, edge, endpoint=True)) ** 2
    taper[:edge] = ramp
    taper[-edge:] = ramp[::-1]
    return taper


def projected_forward_log_ai(depth_m, log_ai, support, highres_interval_m):
    depth_m = np.asarray(depth_m, dtype=np.float64)
    log_ai = np.asarray(log_ai, dtype=np.float64)
    support = np.asarray(support, dtype=bool)
    output = np.full(log_ai.shape, np.nan, dtype=np.float64)
    factor_float = MODEL_GRID_INTERVAL_M / float(highres_interval_m)
    factor = int(round(factor_float))
    if factor < 1 or not np.isclose(factor_float, factor, rtol=0.0, atol=1.0e-6):
        raise ValueError('High-resolution LAS axis is not nested with the 5 m model interval.')
    taps = finite_support_fir(factor)
    for run in finite_runs(support & np.isfinite(log_ai)):
        if run.stop - run.start < 3:
            continue
        model_log_ai, model_support = valid_filter_decimate(log_ai[run], factor=factor, taps=taps)
        model_depth = depth_m[run][::factor]
        model_indices = np.arange(run.start, run.stop, factor, dtype=np.int64)
        if model_log_ai.shape != model_depth.shape or model_depth.shape != model_indices.shape:
            raise ValueError('Projected model arrays have inconsistent shapes.')
        if model_log_ai.size < 2 or not np.any(model_support):
            continue
        velocity = velocity_from_ai(np.exp(model_log_ai), a=AI_VP_A, b=AI_VP_B)
        synthetic = forward_depth(
            model_log_ai, velocity, model_depth, wavelet_time_s, wavelet_amplitude,
            output_chunk_size=FORWARD_OUTPUT_CHUNK_SIZE,
        )
        output[model_indices[model_support]] = synthetic[model_support]
    return output


def seeded_rng(*parts):
    identity = ':'.join(str(part) for part in (RANDOM_SEED, *parts))
    seed = int.from_bytes(hashlib.sha256(identity.encode('utf-8')).digest()[:8], 'little')
    return np.random.default_rng(seed)

In [ ]:
def sample_donor_patch(target_well, candidate, target_depth, rng):
    key = CANDIDATES[candidate]['key']
    span_m = float(target_depth[-1] - target_depth[0])
    donor_names = [name for name in sorted(wells) if name != target_well]
    rng.shuffle(donor_names)
    for donor_name in donor_names:
        donor = wells[donor_name]
        depth = donor['tvdss_m'].astype(np.float64)
        residual = donor[f'{key}_ultrafine'].astype(np.float64)
        support = donor[f'{key}_support'].astype(bool) & np.isfinite(residual)
        runs = [
            run for run in finite_runs(support)
            if depth[run.stop - 1] - depth[run.start] >= span_m
        ]
        if not runs:
            continue
        run = runs[int(rng.integers(0, len(runs)))]
        start_min = float(depth[run.start])
        start_max = float(depth[run.stop - 1] - span_m)
        source_start = start_min if start_max <= start_min else float(rng.uniform(start_min, start_max))
        query = source_start + (target_depth - float(target_depth[0]))
        local = np.interp(query, depth[run], residual[run]).astype(np.float64)
        if np.all(np.isfinite(local)):
            return donor_name, source_start, local
    raise ValueError(f'No donor residual spans {span_m:.1f} m for {target_well} {candidate}.')


def direct_patch_one(target_well, candidate):
    target = wells[target_well]
    config = CANDIDATES[candidate]
    key = config['key']
    depth = target['tvdss_m'].astype(np.float64)
    intervals = np.diff(depth)
    dz_m = float(np.median(intervals))
    if np.any(intervals <= 0.0) or not np.allclose(intervals, dz_m, rtol=1.0e-5, atol=1.0e-6):
        raise ValueError(f'{target_well}: irregular high-resolution axis.')
    support = target[f'{key}_support'].astype(bool)
    full = target['well_log_ai'].astype(np.float64)
    base = target[f'{key}_body_smooth_log_ai'].astype(np.float64)
    own_residual = target[f'{key}_ultrafine'].astype(np.float64)
    pasted_residual = np.zeros_like(full)
    patch_rows = []
    rows = event_metrics.loc[
        event_metrics['well_name'].eq(target_well) & event_metrics['candidate'].eq(candidate)
    ].sort_values('event_rank').head(3)
    if rows.empty:
        raise ValueError(f'{target_well} {candidate}: no published event windows.')
    rng = seeded_rng(target_well, candidate)
    for row in rows.itertuples(index=False):
        event_mask = support & (depth >= float(row.event_top_m)) & (depth <= float(row.event_bottom_m))
        indices = np.flatnonzero(event_mask)
        if indices.size < 3 or np.any(np.diff(indices) != 1):
            raise ValueError(f'{target_well} {candidate}: event support is not contiguous.')
        local_depth = depth[indices]
        donor_name, donor_start, local = sample_donor_patch(target_well, candidate, local_depth, rng)
        local -= float(np.mean(local))
        scale = float(rng.uniform(*PATCH_SCALE_RANGE))
        local *= scale
        sigma_samples = (float(config['fine_fwhm_m']) / 2.354820045) / dz_m
        local -= gaussian_filter1d(local, sigma=sigma_samples, mode='reflect', truncate=4.0)
        local *= edge_taper(local.size)
        pasted_residual[indices] = local
        patch_rows.append({
            'event_rank': int(row.event_rank),
            'top_m': float(row.event_top_m),
            'bottom_m': float(row.event_bottom_m),
            'donor_well': donor_name,
            'donor_start_m': donor_start,
            'scale': scale,
        })
    enhanced = np.where(support, np.clip(base + pasted_residual, LOG_AI_MIN, LOG_AI_MAX), np.nan)
    full_forward = projected_forward_log_ai(depth, full, support, dz_m)
    base_forward = projected_forward_log_ai(depth, base, support, dz_m)
    enhanced_forward = projected_forward_log_ai(depth, enhanced, support, dz_m)
    common = np.isfinite(full_forward) & np.isfinite(base_forward) & np.isfinite(enhanced_forward)
    result = {
        'target_well': target_well, 'candidate': candidate, 'depth_m': depth,
        'support': support, 'full_log_ai': full, 'base_log_ai': base,
        'own_residual': own_residual, 'pasted_residual': pasted_residual,
        'enhanced_log_ai': enhanced, 'full_forward': full_forward,
        'base_forward': base_forward, 'enhanced_forward': enhanced_forward,
        'forward_support': common, 'patch_rows': patch_rows,
        'base_forward_corr': safe_corr(base_forward, full_forward, common),
        'enhanced_forward_corr': safe_corr(enhanced_forward, full_forward, common),
        'forward_delta_rms_ratio': rms(enhanced_forward[common] - base_forward[common]) / rms(full_forward[common]),
        'pasted_residual_rms': rms(pasted_residual[support]),
        'own_residual_rms': rms(own_residual[support]),
    }
    return result


results = {}
metric_rows = []
for target_well in TARGET_WELLS:
    for candidate in CANDIDATES:
        result = direct_patch_one(target_well, candidate)
        results[(target_well, candidate)] = result
        metric_rows.append({
            'target_well': target_well, 'candidate': candidate,
            'base_forward_corr': result['base_forward_corr'],
            'enhanced_forward_corr': result['enhanced_forward_corr'],
            'forward_delta_rms_ratio': result['forward_delta_rms_ratio'],
            'pasted_residual_rms': result['pasted_residual_rms'],
            'own_residual_rms': result['own_residual_rms'],
            'pasted_to_own_rms_ratio': result['pasted_residual_rms'] / result['own_residual_rms'],
        })
        np.savez_compressed(
            ARTIFACT_DIR / f'{target_well}__{candidate}.npz',
            depth_m=result['depth_m'], support=result['support'],
            full_log_ai=result['full_log_ai'], base_log_ai=result['base_log_ai'],
            own_residual=result['own_residual'], pasted_residual=result['pasted_residual'],
            enhanced_log_ai=result['enhanced_log_ai'], full_forward=result['full_forward'],
            base_forward=result['base_forward'], enhanced_forward=result['enhanced_forward'],
            forward_support=result['forward_support'],
        )
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
display(metrics.round(4))

In [ ]:
def plot_result(result):
    depth = result['depth_m']
    patch_rows = result['patch_rows']
    figure, axes = plt.subplots(len(patch_rows), 3, figsize=(10.8, 9.0), sharey='row')
    for row_index, patch in enumerate(patch_rows):
        top, bottom = patch['top_m'], patch['bottom_m']
        margin = max(5.0, 0.12 * (bottom - top))
        visible = (depth >= top - margin) & (depth <= bottom + margin)
        ax_ai, ax_residual, ax_forward = axes[row_index]
        ax_ai.plot(result['full_log_ai'][visible], depth[visible], color='0.45', lw=0.8, label='target full')
        ax_ai.plot(result['base_log_ai'][visible], depth[visible], color='tab:orange', lw=1.5, label='base')
        ax_ai.plot(result['enhanced_log_ai'][visible], depth[visible], color='tab:blue', lw=1.0, label='direct patch')
        ax_residual.plot(result['own_residual'][visible], depth[visible], color='0.55', lw=0.8, label='target residual')
        ax_residual.plot(result['pasted_residual'][visible], depth[visible], color='tab:purple', lw=1.0, label='pasted residual')
        full_visible = visible & np.isfinite(result['full_forward'])
        base_visible = visible & np.isfinite(result['base_forward'])
        enhanced_visible = visible & np.isfinite(result['enhanced_forward'])
        ax_forward.plot(result['full_forward'][full_visible], depth[full_visible], color='black', lw=1.3, label='full forward')
        ax_forward.plot(result['base_forward'][base_visible], depth[base_visible], color='tab:orange', lw=1.3, label='base forward')
        ax_forward.plot(result['enhanced_forward'][enhanced_visible], depth[enhanced_visible], color='tab:blue', lw=1.0, label='patched forward')
        for axis in (ax_ai, ax_residual, ax_forward):
            axis.axhspan(top, bottom, color='mediumpurple', alpha=0.08)
            axis.invert_yaxis()
        ax_residual.set_title(
            f"event {patch['event_rank']} | donor={patch['donor_well']} | scale={patch['scale']:.2f}",
            fontsize=8,
        )
        ax_ai.set_ylabel('TVDSS (m)')
        if row_index == 0:
            ax_ai.legend(fontsize=7, loc='best')
            ax_residual.legend(fontsize=7, loc='best')
            ax_forward.legend(fontsize=7, loc='best')
    axes[0, 0].set_title('log-AI: full / base / patched')
    axes[0, 2].set_title('forward comparison')
    axes[-1, 0].set_xlabel('log-AI')
    axes[-1, 1].set_xlabel('residual log-AI')
    axes[-1, 2].set_xlabel('seismic amplitude')
    figure.suptitle(
        f"{result['target_well']} | {result['candidate']} | direct random well_patch\n"
        f"forward corr: base={result['base_forward_corr']:.4f}, patched={result['enhanced_forward_corr']:.4f}; "
        f"pasted/own residual RMS={result['pasted_residual_rms'] / result['own_residual_rms']:.3f}",
        fontsize=11,
    )
    figure.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
    output = FIGURE_DIR / f"{result['target_well']}__{result['candidate']}__direct_patch.png"
    figure.savefig(output, bbox_inches='tight')
    plt.close(figure)
    return output


figure_paths = {key: plot_result(result) for key, result in results.items()}
publication = {
    'schema': 'well_ultrafine_direct_patch_v1',
    'status': 'completed',
    'source': str(SOURCE_DIR),
    'random_seed': RANDOM_SEED,
    'patch_scale_range': list(PATCH_SCALE_RANGE),
    'candidates': CANDIDATES,
    'target_wells': list(TARGET_WELLS),
    'figures': {f'{well}:{candidate}': str(path) for (well, candidate), path in figure_paths.items()},
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(publication, indent=2), encoding='utf-8')
for candidate in CANDIDATES:
    display(Image(filename=str(figure_paths[('2-ANP-2A-RJS', candidate)])))
print(f'Published: {OUTPUT_DIR}')